# Session 1: repeated loading breaks things that one pull cannot
**A. Real-world problem.** Bend a paper clip once and it stays whole. Bend it back and forth
and it snaps, although no single bend was stronger than the first. Bridges, axles and aircraft
parts fail the same way. This is *fatigue*.

**Plan.** (1) draw a cyclic stress wave, (2) write the simple "life versus stress" relation
(the S-N curve), (3) turn a stress into a number of cycles and a time, (4) plot the curve.
This is a simplified teaching model with illustrative numbers for a generic steel. It is not
a design tool for real parts.

In [ ]:
import pathlib

import numpy as np

OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

SU = 600.0      # ultimate (one-pull) strength of our illustrative steel, MPa
SE = 300.0      # endurance limit = 0.5 * SU, a common rule of thumb for steels, MPa
# Basquin line through two anchor points: 0.9*SU at 1000 cycles and SE at 1 million cycles
B = -np.log10(0.9 * SU / SE) / 3.0            # exponent b (negative); about -0.085
A = (0.9 * SU) ** 2 / SE                      # coefficient A, MPa

## Step 1: the cyclic stress wave
A shaft that turns while it is bent sees every point of its surface go from pull to push and
back again. The stress is a wave: `stress = mean + amplitude * sin(2 pi f t)`.
*Amplitude* is the height of the wave above its middle; one *cycle* is one full wave.

In [ ]:
def stress_wave(t, amplitude, mean=0.0, freq=1.0):
    """Stress (MPa) at time t (s) for a sine load with the given amplitude, mean and frequency (Hz)."""
    return mean + amplitude * np.sin(2 * np.pi * freq * np.asarray(t, dtype=float))


def stress_range(amplitude):
    """Range = highest minus lowest stress of one cycle = 2 * amplitude."""
    return 2.0 * amplitude

## Step 2: the S-N curve (Basquin's relation)
Test many identical bars, each at a different stress amplitude S_a, and count the cycles N
until each breaks. On log axes the points fall near a straight line:

    S_a = A * N**b        (b is negative: more stress, fewer cycles)

Our line passes through 0.9*S_u at 1000 cycles and through the endurance limit S_e at one
million cycles. For many steels, S_e is about half of S_u and b is a small negative number,
about -0.1 (here -0.085). Below S_e a steel part may last "forever" (see the caveat in
Session 2).

In [ ]:
def cycles_to_failure(amplitude):
    """Cycles to failure for a fully reversed stress amplitude (MPa).
    Returns numpy.inf at or below the endurance limit (no predicted failure, steel only)."""
    amp = float(amplitude)
    if amp <= SE:
        return np.inf
    return float((amp / A) ** (1.0 / B))


def time_to_failure_hours(amplitude, freq):
    """Hours of running at `freq` cycles per second until failure (inf if no predicted failure)."""
    return cycles_to_failure(amplitude) / freq / 3600.0

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    t = np.linspace(0, 2, 400)
    fig, ax = plt.subplots(figsize=(7, 3.8))
    ax.plot(t, stress_wave(t, 400), label="stress amplitude 400 MPa")
    ax.axhline(SU, color="red", ls="--", label="one-pull strength 600 MPa")
    ax.axhline(-SU, color="red", ls="--")
    ax.axhline(0, color="gray", lw=0.8)
    ax.annotate("", xy=(0.25, 400), xytext=(0.25, 0), arrowprops=dict(arrowstyle="<->"))
    ax.text(0.28, 180, "amplitude", fontsize=9)
    ax.set_xlabel("time (s), 1 Hz")
    ax.set_ylabel("stress (MPa)")
    ax.set_title("A cyclic stress wave never exceeds the one-pull strength, yet it breaks the part")
    ax.title.set_fontsize(9)
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8, loc="lower right")
    fig.tight_layout()
    fig.savefig(OUT / "stress_wave.png", dpi=120)

    n = np.logspace(3, 7, 200)
    s = np.where(n < 1e6, A * n ** B, SE)
    fig2, ax2 = plt.subplots(figsize=(6.5, 4.2))
    ax2.semilogx(n, s, label="S-N curve (illustrative steel)")
    ax2.axhline(SU, color="red", ls="--", label="one-pull strength S_u")
    ax2.axhline(SE, color="gray", ls=":", label="endurance limit S_e")
    for amp in (500, 400, 350):
        nn = cycles_to_failure(amp)
        ax2.plot(nn, amp, "ko")
        ax2.annotate(f"{amp} MPa: {nn:,.0f} cycles", (nn, amp), textcoords="offset points", xytext=(6, 5), fontsize=8)
        print(f"amplitude {amp} MPa -> {nn:,.0f} cycles")
    ax2.set_xlabel("cycles to failure N (log scale)")
    ax2.set_ylabel("stress amplitude S_a (MPa)")
    ax2.set_title("S-N curve: the more stress, the fewer cycles")
    ax2.grid(alpha=0.3, which="both")
    ax2.legend(fontsize=8)
    fig2.tight_layout()
    fig2.savefig(OUT / "sn_curve.png", dpi=120)
    print(f"b = {B:.4f}, A = {A:.1f} MPa")
    print(f"400 MPa at 10 Hz lasts {time_to_failure_hours(400, 10) * 60:.1f} minutes")

## Where this stops
Real fatigue data scatter widely, and mean stress, notches and surface finish all matter.
Those are Sessions 2 and 3.

## Coding tasks
1. **Easy.** Change the amplitude in `lesson.py` to 450 and 320 MPa. Predict before you run which one lasts longer and by about how much, then compare with `cycles_to_failure`.
2. **Stretch.** Write `amplitude_for_life(n)` that returns the amplitude for a given life n between 1,000 and 1,000,000 cycles, and check that `cycles_to_failure(amplitude_for_life(1e5))` gives back 1e5.
3. **Explain.** Explain in your own words why the S-N curve is drawn with a logarithmic cycle axis, and what a straight line on log axes means.

## Assessment
- **A. Conceptual.** Why can a part break under a repeated stress that is smaller than the stress needed to break it in one pull? What does the part 'remember' from one cycle to the next?
- **B. Numerical.** The same steel is loaded at an amplitude of 350 MPa and 20 Hz. Find the number of cycles to failure and the running time in hours. Check that the answer is sensible.
- **C. Graph and simulation interpretation.** On the S-N graph, the amplitude is raised from 400 MPa to 500 MPa, a rise of 25 percent. By what factor does the life fall? What does the steep curve tell a designer?
- **D. Design and optimisation challenge.** A rotating bar sees an amplitude of 450 MPa. It must run 8 hours a day at 10 Hz for 30 days. (a) How many cycles is that? (b) How long does it last at 450 MPa? (c) In bending, stress goes as 1/d^3 (d = diameter). By what factor must the diameter grow to bring the stress down to the endurance limit?

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    checks.append(("S-N line passes through 0.9*S_u at 1000 cycles", abs(L.A * 1000.0 ** L.B - 540.0) < 1e-6))
    checks.append(("S-N line passes through S_e = 300 MPa at 1 million cycles", abs(L.A * 1e6 ** L.B - 300.0) < 1e-6))
    checks.append((f"exponent b is about -0.085 (got {L.B:.4f}), near the typical -0.1", abs(L.B + 0.0851) < 1e-4))
    n400 = L.cycles_to_failure(400)
    hand = 10 ** (np.log10(400 / 972.0) / L.B)       # hand calculation in logs
    checks.append((f"worked example: 400 MPa gives about 34,000 cycles (got {n400:,.0f})", abs(n400 - 34017) < 5 and abs(n400 / hand - 1) < 1e-9))
    minutes = L.time_to_failure_hours(400, 10) * 60
    checks.append((f"worked example: 34,000 cycles at 10 Hz is about 57 minutes (got {minutes:.1f})", abs(minutes - 56.7) < 0.1))
    checks.append(("more stress means fewer cycles", L.cycles_to_failure(500) < L.cycles_to_failure(400) < L.cycles_to_failure(350)))
    checks.append(("500 MPa gives about 2,500 cycles", abs(L.cycles_to_failure(500) - 2471) < 5))
    checks.append(("below the endurance limit no failure is predicted", L.cycles_to_failure(250) == np.inf and L.cycles_to_failure(300) == np.inf))
    checks.append(("edge: just above the limit the life is finite and a little under 1 million cycles", 7e5 < L.cycles_to_failure(300.5) < 1e6))
    checks.append(("edge: at 0.9*S_u the life is 1000 cycles", abs(L.cycles_to_failure(540) - 1000) < 1e-6))
    t = np.linspace(0, 1, 1001)
    w = L.stress_wave(t, 400)
    checks.append(("wave peaks at +400 and -400 MPa, below S_u = 600", abs(w.max() - 400) < 0.1 and abs(w.min() + 400) < 0.1 and w.max() < L.SU))
    checks.append(("stress range is twice the amplitude (800 MPa)", L.stress_range(400) == 800))
    w2 = L.stress_wave(np.linspace(0, 1, 1001), 100, mean=50)
    checks.append(("a mean of 50 MPa shifts the wave to 150 and -50", abs(w2.max() - 150) < 0.1 and abs(w2.min() + 50) < 0.1))
    # extra numbers used in the text
    checks.append(("350 MPa gives 163,392 cycles; at 20 Hz that is 2.27 hours", abs(L.cycles_to_failure(350) - 163392) < 5 and abs(L.time_to_failure_hours(350, 20) - 2.27) < 0.005))
    checks.append(("450 MPa gives 8,522 cycles, which is 14.2 minutes at 10 Hz", abs(L.cycles_to_failure(450) - 8522) < 5 and abs(L.time_to_failure_hours(450, 10) * 60 - 14.2) < 0.05))
    checks.append(("raising the amplitude from 400 to 500 MPa cuts life by a factor of 13.8", abs(L.cycles_to_failure(400) / L.cycles_to_failure(500) - 13.77) < 0.01))
    checks.append(("8 hours a day at 10 Hz for 30 days is 8.64 million cycles", 8 * 3600 * 10 * 30 == 8640000))
    checks.append(("cutting 450 MPa to 300 MPa in bending needs a diameter 1.145 times larger (stress goes as 1/d^3)", abs((450 / 300) ** (1 / 3) - 1.1447) < 1e-4))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
